#### This Notebook attempts to look at all of the different columns, based on the data of movies with scores and first filters out columns that we no longer need for anything. Then we can set up all of the various encodings

### Here are the columns:
id, title, vote_average, vote_count, status, release_date, revenue, runtime, adult, budget, imdb_id, original_language, original_title, overview, popularity, tagline, genres, production_companies, production_countries, spoken_languages, keywords, actor_avg, actor_med, actor_dev, production_avg, production_med, production_dev

##### Important Categorical features that do not need encoding:
id, title, release_date

##### Important Numerical features that do not need processing:
revenue, runtime, budget, popularity, vote_average, vote_count, actor_avg, actor_med, actor_dev, production_avg, production_med, production_dev

##### Categorical that needs Encoding
adult (binary), original_language, genres, spoken_languages, production_countries (this is a comma seperated list of countries)


#### Potential Encoding Opportunities
release_date -> month, year
original_title -> bool if it matches the title

#### Unimportant/Too difficult to Encode
imdb_id, overview, tagline, status,
production_companies -> seem like too many to encode
keywords -> also a ton, may be too difficult to encode


In [1]:
# inputs
import pandas as pd

In [2]:
# Loads in the data
df = pd.read_csv("movie-data/movies_with_scores.csv")

In [3]:
# Drop columns that are too difficult to encode
cols_to_drop = ['imdb_id', 'overview', 'tagline', 'production_companies', 'keywords', "status"]
df = df.drop(columns=cols_to_drop)
df.head()

,id,title,vote_average,vote_count,release_date,revenue,runtime,adult,budget,original_language,...,popularity,genres,production_countries,spoken_languages,actor_avg,actor_med,actor_dev,production_avg,production_med,production_dev
0,27205,Inception,8.364,34495,7/15/2010,825532764,148,False,160000000,en,...,83.952,"Action, Science Fiction, Adventure","United Kingdom, United States of America","English, French, Japanese, Swahili",338.466820,246.272896,238.147005,1054.563364,1034.781231,39.564266
1,157336,Interstellar,8.417,32571,11/5/2014,701729206,169,False,165000000,en,...,140.241,"Adventure, Drama, Science Fiction","United Kingdom, United States of America",English,258.623745,154.117914,222.651709,862.155016,1034.781231,376.018437
2,155,The Dark Knight,8.512,30619,7/16/2008,1004558444,152,False,185000000,en,...,130.643,"Drama, Action, Crime, Thriller","United Kingdom, United States of America","English, Mandarin",267.925115,191.813811,210.531561,773.341653,836.505176,328.170034
3,19995,Avatar,7.573,29815,12/15/2009,2923706026,162,False,237000000,en,...,79.932,"Action, Adventure, Fantasy, Science Fiction","United States of America, United Kingdom","English, Spanish",625.079911,512.009729,322.765735,1620.772570,1241.115622,759.313896
4,24428,The Avengers,7.710,29166,4/25/2012,1518815515,143,False,220000000,en,...,98.082,"Science Fiction, Action, Adventure",United States of America,"English, Hindi, Russian",883.508364,865.991657,283.031790,750.195662,566.345028,629.099258


In [ ]:
# Convert release_date to datetime and extract year and month
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = df['release_date'].dt.year
df['release_month'] = df['release_date'].dt.month
# Replace empty years with 1900 and empty months with 0
df['release_year'] = df['release_year'].fillna(1900).astype(int)
df['release_month'] = df['release_month'].fillna(0).astype(int)

# Create a new binary column: 1 if original_title matches title, else 0
df['original_title_matches'] = (df['original_title'] == df['title']).astype(int)

print(df.head())
# print(df["original_title_matches"].value_counts())


       id            title  vote_average  vote_count release_date     revenue  \
0   27205        Inception         8.364       34495   2010-07-15   825532764   
1  157336     Interstellar         8.417       32571   2014-11-05   701729206   
2     155  The Dark Knight         8.512       30619   2008-07-16  1004558444   
3   19995           Avatar         7.573       29815   2009-12-15  2923706026   
4   24428     The Avengers         7.710       29166   2012-04-25  1518815515   

   runtime  adult     budget original_language  ...  \
0      148  False  160000000                en  ...   
1      169  False  165000000                en  ...   
2      152  False  185000000                en  ...   
3      162  False  237000000                en  ...   
4      143  False  220000000                en  ...   

                     spoken_languages   actor_avg   actor_med   actor_dev  \
0  English, French, Japanese, Swahili  338.466820  246.272896  238.147005   
1                           

In [9]:
# One-hot encode 'adult'
dummies_adult = pd.get_dummies(df['adult'], prefix='adult', drop_first=False)
print("adult produces", dummies_adult.shape[1], "columns")

# One-hot encode 'original_language'
dummies_language = pd.get_dummies(df['original_language'], prefix='orig_lang', drop_first=False)
print("original_language produces", dummies_language.shape[1], "columns")

# print(dummies_language.head())


adult produces 2 columns
original_language produces 86 columns


In [10]:
def clean_and_get_dummies(series, sep=','):
    # Fill missing values, force string type, and strip whitespace from each token
    s = series.fillna("").astype(str)
    s = s.apply(lambda x: ",".join([token.strip() for token in x.split(sep) if token.strip()]) if x != "" else x)
    return s.str.get_dummies(sep=sep)

# Apply the helper function to each column
dummies_genres = clean_and_get_dummies(df['genres'], sep=',')
print("genres produces", dummies_genres.shape[1], "columns")

dummies_prod_countries = clean_and_get_dummies(df['production_countries'], sep=',')
print("production_countries produces", dummies_prod_countries.shape[1], "columns")

dummies_spoken_langs = clean_and_get_dummies(df['spoken_languages'], sep=',')
print("spoken_languages produces", dummies_spoken_langs.shape[1], "columns")


genres produces 19 columns
production_countries produces 144 columns
spoken_languages produces 136 columns


In [12]:
# Concatenate dummy DataFrames with the original DataFrame
df_encoded = pd.concat([df, dummies_adult, dummies_language, dummies_genres, 
                        dummies_prod_countries, dummies_spoken_langs,], axis=1)

# Drop the original categorical columns that have been encoded
cols_to_remove = ['adult', 'original_language', 'genres', 'production_countries', 
                  'spoken_languages', 'original_title']
df_encoded = df_encoded.drop(columns=cols_to_remove)

df_encoded.head()


,id,title,vote_average,vote_count,release_date,revenue,runtime,budget,popularity,actor_avg,...,Ukrainian,Urdu,Vietnamese,Volapük,Welsh,Wolof,Xhosa,Yiddish,Yoruba,Zulu
0,27205,Inception,8.364,34495,2010-07-15,825532764,148,160000000,83.952,338.466820,...,0,0,0,0,0,0,0,0,0,0
1,157336,Interstellar,8.417,32571,2014-11-05,701729206,169,165000000,140.241,258.623745,...,0,0,0,0,0,0,0,0,0,0
2,155,The Dark Knight,8.512,30619,2008-07-16,1004558444,152,185000000,130.643,267.925115,...,0,0,0,0,0,0,0,0,0,0
3,19995,Avatar,7.573,29815,2009-12-15,2923706026,162,237000000,79.932,625.079911,...,0,0,0,0,0,0,0,0,0,0
4,24428,The Avengers,7.710,29166,2012-04-25,1518815515,143,220000000,98.082,883.508364,...,0,0,0,0,0,0,0,0,0,0


In [14]:
print("Final DataFrame columns:")
print(df_encoded.columns.tolist())

Final DataFrame columns:
['id', 'title', 'vote_average', 'vote_count', 'release_date', 'revenue', 'runtime', 'budget', 'popularity', 'actor_avg', 'actor_med', 'actor_dev', 'production_avg', 'production_med', 'production_dev', 'release_year', 'release_month', 'original_title_matches', 'adult_False', 'adult_True', 'orig_lang_ab', 'orig_lang_af', 'orig_lang_am', 'orig_lang_ar', 'orig_lang_az', 'orig_lang_be', 'orig_lang_bg', 'orig_lang_bm', 'orig_lang_bn', 'orig_lang_bs', 'orig_lang_ca', 'orig_lang_cn', 'orig_lang_cs', 'orig_lang_cy', 'orig_lang_da', 'orig_lang_de', 'orig_lang_dv', 'orig_lang_dz', 'orig_lang_el', 'orig_lang_en', 'orig_lang_es', 'orig_lang_et', 'orig_lang_eu', 'orig_lang_fa', 'orig_lang_fi', 'orig_lang_fr', 'orig_lang_ga', 'orig_lang_gl', 'orig_lang_gu', 'orig_lang_he', 'orig_lang_hi', 'orig_lang_hr', 'orig_lang_hu', 'orig_lang_hy', 'orig_lang_id', 'orig_lang_is', 'orig_lang_it', 'orig_lang_iu', 'orig_lang_ja', 'orig_lang_ka', 'orig_lang_km', 'orig_lang_kn', 'orig_lang_ko'

In [15]:
df_encoded.to_csv("movie-data/movies_encoded.csv", index=False)
print("Saved movies_encoded.csv")

Saved movies_encoded.csv
